# Spatial Proteomics Analysis (SpPrAn)

Welcome to our pipeline's example!    
This notebook is intended to show you how to execute the pipeline or specific functions in a Jupyter Notebook, as well as to visualize resulting plots.

It is important to mention that for executing this notebook, you need to create a copy into repository's root directory.

## Configurations for pipeline

The first thing we suggest to do is adjusting the pipeline configurations to your needs.   
Specifically speaking, you need to adjust the following:

1. **Workspace**: Here, you need to establish the ***input directory*** (where your data file is located), the ***filetype*** of your data (usually is *tsv*, but also pipeline accepts *csv*), and the ***output directory*** (where the results from the pipeline will be saved).

2. Adding a list of **protein markers** that will be used for cell typing (they must appear in the data file columns).

3. Adding a dictionary for estabishing the **cell types** and which protein markers should be listen for defining them.

4. Adding a dictionary of **custom colors** for identifying the cell types.

5. Other configurations related to the pipeline.

***IMPORTANT***:  
* Data file **must** containt a two-character code followed by "_objects" in the name. For example: *Tissue_A1_objects.tsv*, as is shown in `data/example/input`.  

* In addition, data file **must** contain the following columns:

    1. "MV - NUC - {protein_marker}" for each 'protein_marker' in your study.
    
    2. "X-coordinate" and "Y-coordinate", for the spatial plotting.
    
    3. "Positivity - {protein_marker}*" for each 'protein_marker' in your study.

In order to do that, we suggest to duplicate the YAML file located in `config/config_example.yaml` and then edit it for your specific conditions.

In [1]:
!cp config/config_example.yaml config/config.yaml

### How to edit YAML file

#### Option 1: Code editor

You can open `config.yaml` in the code editor of your preference (we suggest **VS Code**) for editing.  
This is how `config.yaml` file looks like:

```
# Add the pathnames to the directories inside quotes, as well as the files' format type (tsv or csv).
workspace:
  input_dir: "/data/example/input"
  output_dir: "/data/example"
  filetype: tsv

# Protein markers should be named exactly as in the "Positivity" columns in the csv files.
# To add a new protein marker, simply add a new line following the format of the previous ones,
#  without forgetting that the protein marker's name must be inside quotes.
protein_markers:
  - "Positivity - Pan-CK* (MV - CYTO)"      # first element
  - "Positivity - CD45* (MV - CYTO)"        # second element
  - "Positivity - Vimentin* (MV - CYTO)"    # third element
  - "Positivity - CD31* (MV - CYTO)"        # ...

# For defining cell types: 
# - Name all the cell types using quotes.
# - For each cell type, generate a list containing only 0, 1, and ~ values.
# - Lists corresponds to the previous protein_markers list and resemble their presense or absent in
#    the defined type of cells.
# - 1 means is protein marker is present, 0 means is abscent, ~ means is not being consider as part of
#    the definition for this cell type.
# - Lists MUST have the same amount of values as proteins in the protein_markers list.
# - To add a new cell type, simply add a new line following the format of the previous ones.
cell_types:
  "C cells": [0, 0, 1, 0]  # [first element, second element, third element, ...]
  "T1 cells": [1, 0, 0, 0]
  "T2 cells": [1, 0, 1, 0]
  "I cells": [0, 1, ~, 0]

# If needed, please establish the HEX color code for each cell type. If not needed, comment or delete this option.
# Cell type's name MUST be the same as in cell_types.
# It MUST contain an "Other cells" name with its corresponding HEX color value.
custom_colors:
  "C cells": "#00aded"
  "I cells": "#ff0000"
  "Other cells": "#a3a3a3"
  "T1 cells": "#fbfb00"
  "T2 cells": "#00ff00"

# Setting this value to True will overwrite previously output files, such as csv files and plots.
overwrite_existing_files: False

# Setting this value to False will not locally save anndata files and it will require to reprocess
#  information each time pipeline is run.
locally_save_anndata_files: True

# Spatial plot adjustments
spatial_plot:
  dpi: 300
  scatter_point_size: 50
```

#### Option 2: Python code

You can also edit `config.yaml` content using Python code.

In [1]:
import yaml

# Read configuration file
with open('config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)
config

{'workspace': {'input_dir': 'data/example/input/',
  'output_dir': 'data/example/',
  'filetype': 'tsv'},
 'protein_markers': ['Positivity - panCK* (MV Cell)',
  'Positivity - CD45hu* (MV Cell)',
  'Positivity - Vim* (MV Cell)',
  'Positivity - FAPa* (MV Cell)'],
 'cell_types': {'C cells': [0, 0, 1, 0],
  'T1 cells': [1, 0, 0, 0],
  'T2 cells': [1, 0, 1, 0],
  'I cells': [0, 1, None, 0]},
 'custom_colors': {'C cells': '#00aded',
  'I cells': '#ff0000',
  'Other cells': '#a3a3a3',
  'T1 cells': '#fbfb00',
  'T2 cells': '#00ff00'},
 'overwrite_existing_files': False,
 'locally_save_anndata_files': True,
 'spatial_plot': {'dpi': 300, 'scatter_point_size': 30}}

In [ ]:
# Modify configuration settings to your needs
config['workspace']['input_dir'] = 'data/example/input/'
config['workspace']['output_dir'] = 'data/example/'
config['workspace']['filetype'] = 'tsv'

protein_markers = [
    'Positivity - panCK* (MV Cell)',
    'Positivity - CD45hu* (MV Cell)',
    'Positivity - Vim* (MV Cell)',
    'Positivity - FAPa* (MV Cell)'
]

# User MUST define their protein markers id variables:
pm_idx_panCK  = 0
pm_idx_CD45hu = 1
pm_idx_Vim    = 2
pm_idx_FAPa   = 3

cell_types_definition = {
    'C cells' : {protein_markers[pm_idx_panCK]:0, protein_markers[pm_idx_CD45hu]:0, protein_markers[pm_idx_Vim]:1,    protein_markers[pm_idx_FAPa]:0},
    'T1 cells': {protein_markers[pm_idx_panCK]:1, protein_markers[pm_idx_CD45hu]:0, protein_markers[pm_idx_Vim]:0,    protein_markers[pm_idx_FAPa]:0},
    'T2 cells': {protein_markers[pm_idx_panCK]:1, protein_markers[pm_idx_CD45hu]:0, protein_markers[pm_idx_Vim]:1,    protein_markers[pm_idx_FAPa]:0},
    'I cells' : {protein_markers[pm_idx_panCK]:0, protein_markers[pm_idx_CD45hu]:1, protein_markers[pm_idx_Vim]:None, protein_markers[pm_idx_FAPa]:0},
}

# Then, save last configurations into the config file
config['protein_markers'] = protein_markers

config['cell_types'] = {
    'C cells' : list(cell_types_definition['C cells' ].values()),
    'T1 cells': list(cell_types_definition['T1 cells'].values()),
    'T2 cells': list(cell_types_definition['T2 cells'].values()),
    'I cells' : list(cell_types_definition['I cells' ].values()),
}

 
# # Alternative option for modifying protein markers and cell types for config file:

# config['protein_markers'] = [
#     'Positivity - panCK* (MV Cell)',
#     'Positivity - CD45hu* (MV Cell)',
#     'Positivity - Vim* (MV Cell)',
#     'Positivity - FAPa* (MV Cell)'
# ]

# config['cell_types'] = {
#     'C cells': [0, 0, 1, 0],
#     'T1 cells': [1, 0, 0, 0],
#     'T2 cells': [1, 0, 1, 0],
#     'I cells': [0, 1, None, 0]
# }


config['custom_colors'] = {
    'C cells': '#00aded',
    'T1 cells': '#fbfb00',
    'T2 cells': '#00ff00',
    'I cells': '#ff0000',
    'Other cells': '#a3a3a3',
}
config['overwrite_existing_files'] = False
config['locally_save_anndata_files'] = True
config['spatial_plot'] = {
    'dpi': 300,
    'scatter_point_size': 30
}

In [3]:
# Write modified configuration back to file
with open('config/config.yaml', 'w') as f:
    yaml.dump(config, f, sort_keys=False)

#### Option 3: HTML file

TBD

## Running the pipeline

Once the YAML file is set with your data, you can execute the pipeline as follows:

In [5]:
!python -m run_pipeline --config config/config.yaml

Generating anndata files for analysis...
Anndata generated!
Anndata saved!
File 'Spatial - Sample A1 (105658 cells).png' created!
File 'Spatial - Sample A1 - C cells (105658 cells).png' created!
File 'Spatial - Sample A1 - I cells (105658 cells).png' created!
File 'Spatial - Sample A1 - T1 cells (105658 cells).png' created!
File 'Spatial - Sample A1 - T2 cells (105658 cells).png' created!
File 'Cell type proportions - Sample A1 (105658 cells).csv' created!


If you want to use Python, you can execute the pipeline as follows:

In [2]:
from spatial_proteomics.core import spatial_proteomics_pipeline

spatial_proteomics_pipeline('config/config.yaml')

Anndata loaded!
File 'Spatial - Sample A1 (105658 cells).png' already exists...
File 'Spatial - Sample A1 - C cells (105658 cells).png' already exists...
File 'Spatial - Sample A1 - I cells (105658 cells).png' already exists...
File 'Spatial - Sample A1 - T1 cells (105658 cells).png' already exists...
File 'Spatial - Sample A1 - T2 cells (105658 cells).png' already exists...
File 'Cell type proportions - Sample A1 (105658 cells)' already exists...
